In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional
from urllib.parse import urljoin

import aiohttp
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Config ───────────────────────────────────────────────────────────────────
INDEX_URL          = "https://man7.org/linux/man-pages/dir_all_alphabetic.html"
OUTPUT_JSONL_PATH  = Path("/scratch4/home/akrik/NTILC/data/man/raw_ai.jsonl")
ERRORS_PATH        = Path("/scratch4/home/akrik/NTILC/data/man/raw_errors_ai.json")
MODEL_ID           = "Qwen/Qwen3.5-9B"

CUDA_VISIBLE_DEVICES  = "5,6"
TENSOR_PARALLEL_SIZE  = 2

MAX_PAGES: Optional[int] = None
REQUEST_TIMEOUT        = 20.0
TEMP                   = 0.0
MAX_NEW_TOKENS         = 2048
MAX_MODEL_LEN          = 194400
HTTP_CONCURRENCY       = 32
INFERENCE_BATCH_SIZE   = 256
RESUME_FROM_DISK       = True

SECTION_1_PATTERN = re.compile(r"\(1\)$")
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/128 Safari/537.36"
    )
}

# Must be set before vLLM loads
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

In [3]:
# ── Prompts ───────────────────────────────────────────────────────────────────
EXTRACTION_SYSTEM_PROMPT = """You extract structured JSON from a Linux man-page HTML body.
Return strict JSON only. Do not include markdown or explanation.
Use this schema exactly:
{
  \"name\": \"string\",
  \"one_line\": \"string\",
  \"description\": \"string\",
  \"invocation\": \"string\",
  \"options\": [
    {
      \"flags\": [\"string\"],
      \"arg\": \"string\",
      \"description\": \"string\"
    }
  ]
}
If a field is unknown, use empty string/list.
Only include real options present in the HTML.
"""

EXAMPLE_OUTPUT = {
    "name": "ls",
    "one_line": "list directory contents",
    "description": "List information about files.",
    "invocation": "ls [OPTION]... [FILE]...",
    "options": [
        {"flags": ["-a", "--all"], "arg": "", "description": "do not ignore entries starting with ."}
    ],
}


def build_prompt(*, source_url: str, html: str) -> str:
    return (
        f"SYSTEM PROMPT:\n{EXTRACTION_SYSTEM_PROMPT}\n\n"
        f"EXAMPLE OUTPUT:\n{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}\n\n"
        f"SOURCE URL:\n{source_url}\n\n"
        "HTML BODY:\n"
        "```html\n"
        f"{html}\n"
        "```\n"
    )

In [4]:
# ── Step 1: Collect links ─────────────────────────────────────────────────────
print("Fetching index ...")
resp = requests.get(INDEX_URL, timeout=30, headers=DEFAULT_HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

all_links: List[str] = sorted({
    urljoin(INDEX_URL, a["href"])
    for a in soup.find_all("a")
    if a.get_text(strip=True) and a.get("href") and SECTION_1_PATTERN.search(a.get_text(strip=True))
})

if MAX_PAGES:
    all_links = all_links[:MAX_PAGES]

print(f"Found {len(all_links)} section-1 links")
all_links[:5]

Fetching index ...
Found 1576 section-1 links


['https://man7.org/linux/man-pages/man1/AS.1.html',
 'https://man7.org/linux/man-pages/man1/Firecfg.1.html',
 'https://man7.org/linux/man-pages/man1/Firejail.1.html',
 'https://man7.org/linux/man-pages/man1/Firemon.1.html',
 'https://man7.org/linux/man-pages/man1/PCPCompat.1.html']

In [5]:
# ── Step 2: Resume ────────────────────────────────────────────────────────────
def load_done_urls(path: Path) -> set[str]:
    done = set()
    if path.exists():
        with path.open() as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if "source_url" in obj:
                        done.add(obj["source_url"])
                except json.JSONDecodeError:
                    pass
    return done


done_urls: set[str] = load_done_urls(OUTPUT_JSONL_PATH) if RESUME_FROM_DISK else set()
todo_links = [u for u in all_links if u not in done_urls]
print(f"{len(done_urls)} already done — {len(todo_links)} remaining")

0 already done — 1576 remaining


In [6]:
# ── Step 3: Fetch all HTML concurrently ───────────────────────────────────────
async def fetch_one(
    session: aiohttp.ClientSession,
    url: str,
    semaphore: asyncio.Semaphore,
) -> tuple[str, str | None]:
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)) as r:
                r.raise_for_status()
                return url, await r.text()
        except Exception as exc:
            print(f"[fetch error] {url}: {exc}")
            return url, None


async def fetch_all(urls: List[str]) -> Dict[str, str | None]:
    semaphore = asyncio.Semaphore(HTTP_CONCURRENCY)
    connector = aiohttp.TCPConnector(limit=HTTP_CONCURRENCY, ssl=False)
    async with aiohttp.ClientSession(headers=DEFAULT_HEADERS, connector=connector) as session:
        tasks = [fetch_one(session, url, semaphore) for url in urls]
        results = {}
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Fetching HTML"):
            url, html = await coro
            results[url] = html
    return results


try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

html_map = asyncio.run(fetch_all(todo_links))
print(f"Fetched {sum(v is not None for v in html_map.values())} pages successfully")

Fetching HTML:   0%|          | 0/1576 [00:00<?, ?it/s]

Fetching HTML: 100%|██████████| 1576/1576 [00:06<00:00, 250.72it/s]

Fetched 1576 pages successfully


In [7]:
# ── Step 4: Build prompts ─────────────────────────────────────────────────────
pairs: List[tuple[str, str]] = []
fetch_errors: List[dict] = []

for url in todo_links:
    html = html_map.get(url)
    if html is None:
        fetch_errors.append({"url": url, "error": "fetch_failed"})
        continue
    pairs.append((url, build_prompt(source_url=url, html=html)))

print(f"Built {len(pairs)} prompts — {len(fetch_errors)} fetch errors")

Built 1576 prompts — 0 fetch errors


In [8]:
# ── Step 5: Load model with vLLM ─────────────────────────────────────────────
# NOTE: To change CUDA_VISIBLE_DEVICES or max_model_len you must restart the kernel.
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.92,
    enable_chunked_prefill=True,
    max_num_batched_tokens=32768,
    max_model_len=MAX_MODEL_LEN,
)

sampling_params = SamplingParams(
    temperature=TEMP,
    max_tokens=MAX_NEW_TOKENS,
)

INFO 02-25 22:00:49 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 194400, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.92, 'max_num_batched_tokens': 32768, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen3.5-9B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-25 22:00:49 [model.py:541] Resolved architecture: Qwen3MoeForCausalLM
INFO 02-25 22:00:49 [model.py:1561] Using max model len 194400


2026-02-25 22:00:49,621	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-25 22:00:49 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 02-25 22:00:49 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=4106937) INFO 02-25 22:00:50 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='Qwen/Qwen3.5-9B', speculative_config=None, tokenizer='Qwen/Qwen3.5-9B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=194400, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoni

...

(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:09 [default_loader.py:291] Loading weights took 10.02 seconds
(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:10 [gpu_model_runner.py:4130] Model loading took 28.51 GiB memory and 10.978764 seconds
(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:18 [backends.py:812] Using cache directory: /home/akrik/.cache/vllm/torch_compile_cache/620ba41090/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:18 [backends.py:872] Dynamo bytecode transform time: 8.42 s
(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) WARNING 02-25 22:01:23 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=384,device_name=NVIDIA_H100_80GB_HBM3.json
(EngineCore_DP0 pid=4106937) (E

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:06<00:00,  8.42it/s]
Capturing CUDA graphs (decode, FULL):  94%|█████████▍| 48/51 [00:04<00:00, 11.04it/s]

(EngineCore_DP0 pid=4106937) (Worker_TP1 pid=4106953) INFO 02-25 22:01:37 [custom_all_reduce.py:216] Registering 9894 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:04<00:00, 10.86it/s]


(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:37 [custom_all_reduce.py:216] Registering 9894 cuda graph addresses
(EngineCore_DP0 pid=4106937) (Worker_TP0 pid=4106951) INFO 02-25 22:01:37 [gpu_model_runner.py:5063] Graph capturing finished in 12 secs, took 1.38 GiB
(EngineCore_DP0 pid=4106937) INFO 02-25 22:01:37 [core.py:272] init engine (profile, create kv cache, warmup model) took 27.61 seconds
INFO 02-25 22:01:40 [llm.py:343] Supported tasks: ['generate']


In [9]:
# ── Step 6: Filter prompts that are too long ──────────────────────────────────
# Must run AFTER Step 5 since we need llm.get_tokenizer()
MAX_PROMPT_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS

tokenizer = llm.get_tokenizer()

errors: List[dict] = list(fetch_errors)  # seed with fetch errors
filtered_pairs, skipped = [], []

for url, prompt in tqdm(pairs, desc="Checking token lengths"):
    n = len(tokenizer.encode(prompt))
    if n <= MAX_PROMPT_TOKENS:
        filtered_pairs.append((url, prompt))
    else:
        print(f"[skip] {url} — {n:,} tokens")
        skipped.append({"url": url, "error": f"prompt_too_long_{n}_tokens"})

errors.extend(skipped)
pairs = filtered_pairs
print(f"{len(pairs)} within limit, {len(skipped)} skipped")

Checking token lengths:  21%|██        | 329/1576 [00:03<00:26, 47.23it/s] 

[skip] https://man7.org/linux/man-pages/man1/g++.1.html — 322,913 tokens


Checking token lengths:  22%|██▏       | 341/1576 [00:04<00:39, 31.43it/s]

[skip] https://man7.org/linux/man-pages/man1/gcc.1.html — 323,294 tokens


Checking token lengths: 100%|██████████| 1576/1576 [00:17<00:00, 92.00it/s] 

1574 within limit, 2 skipped


In [10]:
# ── Step 7: Inference + save ──────────────────────────────────────────────────
def parse_json_output(raw: str) -> dict:
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if m:
        raw = m.group(0)
    return json.loads(raw)


OUTPUT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
ERRORS_PATH.parent.mkdir(parents=True, exist_ok=True)

total_processed = 0

with OUTPUT_JSONL_PATH.open("a") as out_file:
    for batch_start in tqdm(range(0, len(pairs), INFERENCE_BATCH_SIZE), desc="Inference batches"):
        batch         = pairs[batch_start : batch_start + INFERENCE_BATCH_SIZE]
        urls_batch    = [u for u, _ in batch]
        prompts_batch = [p for _, p in batch]

        t0 = time.time()
        outputs = llm.generate(prompts_batch, sampling_params)
        elapsed = time.time() - t0
        print(f"  batch {batch_start // INFERENCE_BATCH_SIZE}: "
              f"{len(batch)} pages in {elapsed:.1f}s ({elapsed/len(batch):.2f}s/page)")

        for url, output in zip(urls_batch, outputs):
            raw_text = output.outputs[0].text
            try:
                parsed = parse_json_output(raw_text)
                parsed["source_url"] = url
                out_file.write(json.dumps(parsed, ensure_ascii=False) + "\n")
            except Exception as exc:
                errors.append({"url": url, "error": str(exc), "raw": raw_text[:2000]})

        out_file.flush()
        total_processed += len(batch)

with ERRORS_PATH.open("w") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\nDone. {total_processed} processed, {len(errors)} errors.")
print(f"Results → {OUTPUT_JSONL_PATH}")
print(f"Errors  → {ERRORS_PATH}")

Inference batches:  14%|█▍        | 1/7 [01:41<10:07, 101.19s/it]

  batch 0: 256 pages in 101.2s (0.40s/page)


Inference batches:  29%|██▊       | 2/7 [03:41<09:23, 112.65s/it]

  batch 1: 256 pages in 120.6s (0.47s/page)


Inference batches:  43%|████▎     | 3/7 [05:27<07:17, 109.40s/it]

...

  batch 4: 256 pages in 76.6s (0.30s/page)


Inference batches:  86%|████████▌ | 6/7 [10:05<01:38, 98.47s/it]

  batch 5: 256 pages in 108.1s (0.42s/page)


Inference batches: 100%|██████████| 7/7 [10:27<00:00, 89.59s/it]

  batch 6: 38 pages in 21.8s (0.57s/page)

Done. 1574 processed, 206 errors.
Results → /scratch4/home/akrik/NTILC/data/man/raw_ai.jsonl
Errors  → /scratch4/home/akrik/NTILC/data/man/raw_errors_ai.json
